# Homework 6 - Bayesian Model Fitting, Fast Fourier Transforms, and Interpolating Reaction Rates (10 pt)

When working with data in astrophysics, one is typically doing one of three tasks: fitting a physical model to data, finding the frequencies hidden in a time series, and looking numbers calculated from observations or detailed calculations that have been tabulated in advance. In this homework assignment we will do one of each. We will first fit a Doppler-broadened (Gaussian) line profile to a dataset, using least squares to find the best fit and then Markov-Chain Monte Carlo to find the full posteriors. We will then use the Fast Fourier Transform to find the oscillation frequencies of a Sun-like star observed by the *Kepler* space telescope, whose oscillation frequencies are used to constrain the stellar mass, radius, and age. Finally, we will interpolate a tabulated version of the triple-$\alpha$ nuclear reaction rate.

**Instructions:** Read the descriptor for each problem, then write your solution code in the Jupyter code cell. Add your homework 6 assignment to the list of things to modify with `git add HW6.ipynb`, and commit changes in your assignment as frequently as you save your homework assignment. Once you are done, commit your finishing changes with the command `git commit -m "Completed HW 6"`, then push/submit to the instructor/TA using `git push`. If you push your assignment more than once, this is fine! Your most recently pushed assignment will be the one that is graded.

To accept this assignment, click the link below and follow the steps:\
https://classroom50.org/PsuAstro410/410astro26/assignments/homework-6/accept

This homework assignment made use of material from the following sources:\
https://zingale.github.io/computational_astrophysics/ffts/fft.html \
https://zingale.github.io/computational_astrophysics/ffts/application-lightcurves.html \
https://zingale.github.io/computational_astrophysics/interpolation/lagrange-interpolation.html \
https://zingale.github.io/computational_astrophysics/homework/homework6.html \
https://spacetelescope.github.io/mast_notebooks/notebooks/Kepler/creating_periodograms/creating_periodograms.html \
https://lightkurve.github.io/lightkurve/tutorials/3-science-examples/asteroseismology-estimating-mass-and-radius.html

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize

# To make the plots pretty
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11

# Problem 1 - Fitting a Doppler-Broadened Line Profile (4 pt)

Spectral lines are never infinitely narrow. In a hot gas, the absorbing atoms move with a Maxwell-Boltzmann distribution of velocities along our line of sight, so each atom absorbs at a slightly Doppler-shifted frequency. This broadens the line into a Gaussian profile
$$
\phi(\nu) = \frac{1}{\alpha_D} \sqrt{\frac{\ln 2}{\pi}} \exp\left[ -\ln 2 \, \frac{(\nu - \nu_0)^2}{\alpha_D^2} \right],
$$
where $\nu_0$ is the line center and $\alpha_D$ is the half-width at half-maximum of the line, which is set by the temperature of the gas. The profile is normalized so that $\int \phi(\nu)\, d\nu = 1$. In this problem, we fit this profile to a measured line, with the two free parameters

$$
\vec{\theta} = (\alpha_D, \nu_0).
$$

We will do this in two steps. First, we use the nonlinear least-squares routine `optimize.leastsq` from Lecture 11 to quickly find the best-fit parameters. Then, we use the Markov-Chain Monte Carlo (MCMC) code from Lecture 12, started from the least-squares solution, to calculate the full posterior distributions of $\alpha_D$ and $\nu_0$.

Download the dataset by making sure that you are in the directory with your homework, and typing into your terminal the command
```bash
curl -O https://raw.githubusercontent.com/PsuAstro410/psuastro410.github.io/main/homeworks/gaussian_line.txt
```
The file has three columns: the frequency $\nu$, the measured line profile $\phi$, and the uncertainty $\sigma_\phi$ in each measurement. Load it with the commands
```python
data = np.loadtxt("gaussian_line.txt")
nu = data[:, 0]
phi = data[:, 1]
phi_err = data[:, 2]
```

We thank Cindy Yuexing Gulis for the idea, scaffolding, and data file which went into the design of this problem.

## Problem 1(a) - Load the data and define the model (1 pt)

Load the dataset, and plot $\phi$ versus $\nu$ with error bars using `ax.errorbar`. Then write a function
```python
def doppler(nu, theta)
```
that returns $\phi(\nu)$, where `theta = [alpha_D, nu_0]`. This is the same form as the function `model(t, theta)` from Lecture 12, so it can be passed directly to the MCMC code later on. From your plot, estimate by eye the values of $\nu_0$ and $\alpha_D$; we will use these as an initial guess in Problem 1(b).

In [ ]:
## Answer here


## Problem 1(b) - Best fit with least squares (1 pt)

Following Lecture 11, write a residual function
```python
def resid(theta, nu, y, yerr)
```
that returns the error-weighted distance between the data and the model, $(y_i - \phi(\nu_i; \vec{\theta}))/\sigma_i$. Use `optimize.leastsq` with your initial guess from Problem 1(a) to find the best-fit values of $\alpha_D$ and $\nu_0$, and print them. Over-plot the best-fit line profile on top of the data.

In [ ]:
## Answer here


## Problem 1(c) - Priors and log-likelihood for the MCMC (1 pt)

We will use the MCMC code from Lecture 12 to compute the posteriors of $\alpha_D$ and $\nu_0$. We have modified the functions to also take the y-errors, which we call $\sigma$.

In [ ]:
def move(func, x, y, theta, sigma, h=0.1, TINY=1e-30):
    # random step in parameter space (uniform "ball" of half-width h)
    dtheta = (2*np.random.rand(theta.size) - 1.) * h
    thetanew = theta + dtheta

    # compute the priors
    p1 = priors(theta)
    p2 = priors(thetanew)
    if p2 <= 0:
        return theta, False
    p = p2 / (p1 + TINY)

    # compute the likelihoods
    logL1 = loglikelihood(func, x, y, theta, sigma)
    logL2 = loglikelihood(func, x, y, thetanew, sigma)

    # probability to accept the new step
    rho = min(np.exp(logL2 - logL1) * p, 1)

    if np.random.rand() < rho:
        return thetanew, True
    return theta, False

def mcmc(func, x, y, theta, sigma, burn_in=1000, MAX_CHAIN=500000):
    thetachain = []
    i = 0
    for _ in range(MAX_CHAIN):
        theta, accepted_move = move(func, x, y, theta, sigma)
        if accepted_move:
            i += 1
            if i > burn_in:
                thetachain.append(theta)
    return np.array(thetachain)

Before we run the MCMC, we need a prior and a likelihood.

Write a function `priors(theta)` that gives flat priors with $0 < \alpha_D \le 100$ and $0 \le \nu_0 \le 100$. It should return 1 if `theta` is inside these bounds, and 0 otherwise. In Lecture 12, the log-likelihood assumed every data point had the same error $\sigma = 1$. Our data has a different error $\sigma_i$ for every point. Also write a new log-likelihood
  ```python
  def loglikelihood(func, x, y, theta, sigma)
  ```
  that returns
  $$
  \log p(\vec{y}; \vec{\theta}) = -\sum_i \left( \frac{y_i - \phi(\nu_i; \vec{\theta})}{\sigma_i} \right)^2 = -\chi^2.
  $$

In [ ]:
## Answer here


## Problem 1(d) - Posteriors of $\alpha_D$ and $\nu_0$ (1 pt)

Run your MCMC code with `MAX_CHAIN=100000` and `burn_in=1000`, using the least-squares solution from Problem 1(b) as the starting point of the chain. Then make a two-dimensional histogram of the chain with `plt.hist2d`, and one-dimensional histograms of $\alpha_D$ and $\nu_0$, as was done in Lecture 12.

In [ ]:
## Answer here


# Problem 2 - The Oscillation Spectrum of a Sun-like Star with the Fast Fourier Transform (3 pt)

Turbulent convection near the surface continuously excites pressure waves that are trapped inside the star, and the star oscillates in thousands of these modes at once. Each mode makes the star's brightness change by only a few parts per million (ppm), but with the extremely precise photometry of the *Kepler* space telescope we can see them in the star's Fourier spectrum.

The p-mode frequencies have a very regular pattern. The properties of these $p$-modes are often used to measure the masses, radii, and even ages of tens of thousands of stars, including many exoplanet host stars.

In this problem, we carry out the first step of such an analysis: computing the oscillation spectrum of the Sun-like star KIC 10963065. The cell below should download the "KIC 10963065-like" data, modified so that we can analyze it with the FFT algorithm that we introduced in class.

In [ ]:
# Uploading flux data for KIC 10963065, slightly modified so that the algorithms we discussed in-class can analyze the data
# t - time in seconds, length power of 2^N
# f_n - Fractional flux change from the mean in parts per million, or 1e6*(flux-flux_mean)/flux_mean
t_n, f_n = np.loadtxt("KIC_10963065_pseudodata.txt")
day = 86400.0    # day in s

## Problem 2(a) - Plot light curve for the FFT (1 pt)

Before we analyze the data, plot the time on the x-axis, and flux variation on the y-axis. Appreciate the wiggles. **Note:** Plot the time in days, not seconds, which it is saved as in `t_n`.

In [ ]:
## Answer here


## Problem 2(b) - The FFT and the power spectrum (1 pt)

Below is the recursive function `fft(f_n)` from lecture: 

In [ ]:
def fft(f_n):
    """ perform a discrete Fourier transform.  We use the same
        conventions as NumPy's FFT
        here,
              N-1         -2 pi i n k /N
        F_k = sum  f   exp
              i=0   n
    """
    N = len(f_n)
    if N == 1:
        return f_n
    else:
        # split into even and odd and find the FFTs of each half
        f_even = f_n[0:N:2]
        f_odd = f_n[1:N:2]

        F_even = fft(f_even)
        F_odd = fft(f_odd)

        # combine them.  Each half has N/2 wavenumbers, but due to
        # periodicity, we can compute N wavenumbers
        omega = np.exp(-2*np.pi*1j/N)

        F_k = np.zeros((N), dtype=np.complex128)
        oterm = omega**np.arange(N//2)
        F_k[0:N//2] = F_even + oterm * F_odd
        F_k[N//2:] = F_even - oterm * F_odd

    return F_k

Compute the FFT $F_k$ of your uniformly sampled light curve from Problem 2(a). For real data, we only need the wavenumbers $k = 0, 1, \ldots, N/2$, which correspond to the physical frequencies
$$
\nu_k = \frac{k}{N \Delta t},
$$
e.g. we stop at the Nyquist frequency. Convert these to $\mu$Hz, and compute the power spectrum
$$
P_k = \left(\frac{2 |F_k|}{N}\right)^2 \quad ({\rm ppm}^2),
$$
which is normalized so that a sinusoid of amplitude $A$ (in ppm) gives a peak of height $A^2$. Plot $P_k$ versus $\nu_k$ on a log-log plot. In addition to a decline in power as $\nu_k$ increases, should see a "hump" of excess power near $\sim$200-300 $\mu$Hz: these are the star's oscillations that constrain its mass, radius, and sometimes even its age.

In [ ]:
## Answer here


## Problem 2(c) - Zooming in near $\nu_{\rm max}$ (1 pt)

Zoom in near the "hump" at higher frequencies (between 100 $\mu$Hz to 4000 $\mu$Hz), and change the x-axis to linear. The broad shape of this hump, and its specific values, is what is asteroseismically useful.

In [ ]:
## Answer here


# Problem 3 - Interpolating a Tabulated Triple-$\alpha$ Reaction Rate (3 pt)

Stellar evolution and hydrodynamics codes need nuclear reaction rates millions of times per time step. Rather than evaluate an expensive formula (or a measured rate) every time, these codes usually tabulate the rate at a set of temperatures ahead of time, and interpolate the table when they need a value. In this problem, we will interpolate a tabulated version of the triple-$\alpha$ rate, which converts helium into carbon in the cores of red giants:
$$
q_{3\alpha} = 5.09\times 10^{11} \rho^2 Y^3 T_8^{-3} e^{-44.027/T_8}~\mathrm{erg~g^{-1}~s^{-1}},
$$
where $T_8 = T/(10^8~{\rm K})$. We focus on the temperature-dependent part of the rate,
$$
\lambda(T) = 5.09\times 10^{11} T_8^{-3} e^{-44.027/T_8}.
$$

Because $\lambda$ varies over 14 orders of magnitude, it is tabulated as $\log_{10} \lambda$ versus $\log_{10} T$ (here $\log$. Below are taublations of $(\log_{10} T)_i$ and $(\log_{10} \lambda)_i$ at 25 temperatures, equally spaced in $\log_{10} T$ between $10^8$ K and $5 \times 10^9$ K.

In [ ]:
logTs = np.array([8.        , 8.07079042, 8.14158083, 8.21237125, 8.28316167, 8.35395208,
                  8.4247425 , 8.49553292, 8.56632333, 8.63711375, 8.70790417, 8.77869459,
                  8.849485  , 8.92027542, 8.99106584, 9.06185625, 9.13264667, 9.20343709,
                  9.2742275 , 9.34501792, 9.41580834, 9.48659875, 9.55738917, 9.62817959,
                  9.69897   ])
loglams = np.array([-7.41396537, -4.75040182, -2.51940655, -0.65591712,  0.89534288,
                     2.18133576,  3.24196026,  4.11111394,  4.81759586,  5.38587342,
                     5.83673388,  6.18783791,  6.45418983,  6.64853718,  6.78171014,
                     6.86290992,  6.89995379,  6.89948323,  6.86714076,  6.80772024,
                     6.72529445,  6.62332361,  6.50474749,  6.37206368,  6.22739411])

In this problem we fit a quadratic (second-order) Lagrange polynomial three tabulated points closest to the temperature $T_0$ where we want the rate.

## Problem 3(a) - Finding the 3 closest points (1 pt)

Write a function
```python
def find_index(logTs, logT0)
```
that returns the index $j$ such that $(\log T)_j$, $(\log T)_{j+1}$, $(\log T)_{j+2}$ are the 3 tabulated points closest to $\log T_0$. Usually the middle point $(\log T)_{j+1}$ is the point closest to $\log T_0$, so there is one point on either side. Near the ends of the table, this is not possible, and you will need to use the 3 points at the very start or very end of the table.

Test your function for $\log_{10} T_0 = 8.01$, $8.8$, and $9.69$, and check by comparing with `logTs` that it gives the right indices. Print your functions evaluations for these three evaluations of $T_0$, and verify the index $j$, and three closest tabulated points used from `logTs`, are all correct.

In [ ]:
## Answer here


## Problem 3(b) - Quadratic Lagrange interpolation (1 pt)

From lecture, the Lagrange polynomial through the 3 points $(x_0, f_0)$, $(x_1, f_1)$, $(x_2, f_2)$ is

$$
f(x) = \sum_{m=0}^{2} \ell_m(x) f_m, \qquad \ell_m(x) = \prod_{n=0, n\ne m}^{2} \frac{x - x_n}{x_m - x_n}.
$$

Write a function
```python
def lagrange_quad(x0, xv, fv)
```
that evaluates this quadratic at the point `x0`, given arrays `xv` and `fv` of the 3 points closest to `x0`. You may start from the code in lecture, but do not use a library (e.g. `scipy.interpolate`). Then write a function
```python
def interp_rate(logT0, logTs, loglams)
```
that uses `find_index` and `lagrange_quad` to return the interpolated $\log_{10} \lambda$ at $\log_{10} T_0$.

Plot the 25 tabulated points, your interpolated $\log_{10} \lambda$ evaluated at 500 values of $\log_{10} T$ between 8 and $\log_{10} (5\times10^9)$, and the exact $\log_{10} \lambda(T)$ from the formula above, all on the same plot.

In [ ]:
## Answer here


## Problem 3(c) - How accurate is the interpolation? (1 pt)

Use a random number generator to pick 25 values uniformly distributed in $\log_{10} T$ between $10^8$ K and $5\times 10^9$ K. For each one:

* Interpolate the table to find $\lambda_0 = 10^{\log\lambda_0}$.
* Compute the exact value $\lambda(T_0)$ from the formula, and the relative error $|\lambda_0 - \lambda(T_0)|/\lambda(T_0)$.

Print $T_0$, the exact rate, the interpolated rate, and the relative error for each temperature.

In [ ]:
## Answer here
